<a href="https://colab.research.google.com/github/nithyanchris-source/ML-Final-Lab-Group-07/blob/main/ML_miniProject7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
from google.colab import files

uploaded = files.upload()

Saving aclImdb_v1.tar.gz to aclImdb_v1.tar.gz


In [5]:
import tarfile
import os

archive = "aclImdb_v1.tar.gz"
extract_path = "/content/data/raw"

os.makedirs(extract_path, exist_ok=True)

print("Extracting IMDb dataset...")

with tarfile.open(archive, "r:gz") as tar:
    tar.extractall(path=extract_path)

print("Extraction complete.")

dataset_path = "/content/data/raw/aclImdb"

print("Dataset exists:", os.path.exists(dataset_path))


Extracting IMDb dataset...


/tmp/ipykernel_1930/600617743.py:12: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=extract_path)


Extraction complete.
Dataset exists: True


In [8]:
import os

folders = {
    "train/pos": os.path.join(dataset_path, "train", "pos"),
    "train/neg": os.path.join(dataset_path, "train", "neg"),
    "train/unsup": os.path.join(dataset_path, "train", "unsup"),
    "test/pos": os.path.join(dataset_path, "test", "pos"),
    "test/neg": os.path.join(dataset_path, "test", "neg"),
}

print("=" * 60)
print("IMDb DATASET VALIDATION")
print("=" * 60)

for name, path in folders.items():
    count = len(os.listdir(path))
    print(f"{name:15} : {count:,} files")

print("=" * 60)

IMDb DATASET VALIDATION
train/pos       : 12,500 files
train/neg       : 12,500 files
train/unsup     : 50,000 files
test/pos        : 12,500 files
test/neg        : 12,500 files


In [6]:
import pandas as pd

def load_reviews(folder, label):
    records = []

    for filename in os.listdir(folder):
        filepath = os.path.join(folder, filename)

        if not os.path.isfile(filepath):
            continue

        try:
            with open(filepath, "r", encoding="utf-8") as f:
                text = f.read().strip()

            records.append({
                "text": text,
                "label": label
            })

        except Exception as e:
            print(f"Error reading {filepath}: {e}")

    return records

In [7]:
train_pos = load_reviews(
    os.path.join(dataset_path, "train", "pos"), 1
)

train_neg = load_reviews(
    os.path.join(dataset_path, "train", "neg"), 0
)

test_pos = load_reviews(
    os.path.join(dataset_path, "test", "pos"), 1
)

test_neg = load_reviews(
    os.path.join(dataset_path, "test", "neg"), 0
)

train_df = pd.DataFrame(train_pos + train_neg)
test_df = pd.DataFrame(test_pos + test_neg)

print("Train:", train_df.shape)
print("Test :", test_df.shape)

Train: (25000, 2)
Test : (25000, 2)


In [11]:
def quality_report(df, name):
    print("=" * 60)
    print(f"DATA QUALITY REPORT — {name}")
    print("=" * 60)

    print("Total rows       :", len(df))
    print("Positive         :", (df["label"] == 1).sum())
    print("Negative         :", (df["label"] == 0).sum())
    print("Missing text     :", df["text"].isna().sum())
    print("Empty text       :", (df["text"].str.strip() == "").sum())
    print("Duplicate rows   :", df["text"].duplicated().sum())
    print("Unique texts     :", df["text"].nunique())

    print("=" * 60)


quality_report(train_df, "TRAIN")
quality_report(test_df, "TEST")

DATA QUALITY REPORT — TRAIN
Total rows       : 25000
Positive         : 12500
Negative         : 12500
Missing text     : 0
Empty text       : 0
Duplicate rows   : 96
Unique texts     : 24904
DATA QUALITY REPORT — TEST
Total rows       : 25000
Positive         : 12500
Negative         : 12500
Missing text     : 0
Empty text       : 0
Duplicate rows   : 199
Unique texts     : 24801


In [8]:
train_texts = set(train_df["text"])
test_texts = set(test_df["text"])

overlap = train_texts.intersection(test_texts)

print("=" * 60)
print("TRAIN / TEST LEAKAGE CHECK")
print("=" * 60)

print("Unique train reviews :", len(train_texts))
print("Unique test reviews  :", len(test_texts))
print("Overlapping reviews  :", len(overlap))

print("=" * 60)

TRAIN / TEST LEAKAGE CHECK
Unique train reviews : 24904
Unique test reviews  : 24801
Overlapping reviews  : 123


In [9]:
os.makedirs("/content/data/processed", exist_ok=True)

train_df.to_csv(
    "/content/data/processed/train_raw.csv",
    index=False
)

test_df.to_csv(
    "/content/data/processed/test_raw.csv",
    index=False
)

print("Saved:")
print("train_raw.csv")
print("test_raw.csv")

Saved:
train_raw.csv
test_raw.csv


In [10]:
# Detailed duplicate and leakage investigation

print("=" * 60)
print("DETAILED DUPLICATE & LEAKAGE INVESTIGATION")
print("=" * 60)

# ---------- TRAIN DUPLICATES ----------
train_dup_mask = train_df["text"].duplicated(keep=False)
train_dups = train_df[train_dup_mask]

print("\nTRAIN")
print("-" * 40)
print("Duplicate rows:", train_df["text"].duplicated().sum())
print("Unique duplicated texts:", train_dups["text"].nunique())

# Check conflicting labels
train_conflicts = (
    train_dups.groupby("text")["label"]
    .nunique()
)

train_conflicting = (train_conflicts > 1).sum()

print("Conflicting-label duplicate texts:", train_conflicting)


# ---------- TEST DUPLICATES ----------
test_dup_mask = test_df["text"].duplicated(keep=False)
test_dups = test_df[test_dup_mask]

print("\nTEST")
print("-" * 40)
print("Duplicate rows:", test_df["text"].duplicated().sum())
print("Unique duplicated texts:", test_dups["text"].nunique())

test_conflicts = (
    test_dups.groupby("text")["label"]
    .nunique()
)

test_conflicting = (test_conflicts > 1).sum()

print("Conflicting-label duplicate texts:", test_conflicting)


# ---------- TRAIN / TEST OVERLAP ----------
train_unique = train_df.drop_duplicates("text")
test_unique = test_df.drop_duplicates("text")

overlap_texts = set(train_unique["text"]) & set(test_unique["text"])

print("\nTRAIN vs TEST")
print("-" * 40)
print("Overlapping unique review texts:", len(overlap_texts))


# Check conflicting labels across train/test
train_overlap = train_unique[
    train_unique["text"].isin(overlap_texts)
].set_index("text")

test_overlap = test_unique[
    test_unique["text"].isin(overlap_texts)
].set_index("text")

common_index = train_overlap.index.intersection(test_overlap.index)

conflicting_overlap = (
    train_overlap.loc[common_index, "label"]
    != test_overlap.loc[common_index, "label"]
).sum()

print("Conflicting-label overlaps:", conflicting_overlap)


# ---------- SHOW EXAMPLES ----------
print("\nEXAMPLE OVERLAPPING REVIEWS")
print("-" * 40)

for i, text in enumerate(list(overlap_texts)[:3], 1):
    train_label = train_overlap.loc[text, "label"]
    test_label = test_overlap.loc[text, "label"]

    print(f"\nExample {i}")
    print("Train label:", train_label)
    print("Test label :", test_label)
    print("Text:", text[:250], "...")

print("\n" + "=" * 60)

DETAILED DUPLICATE & LEAKAGE INVESTIGATION

TRAIN
----------------------------------------
Duplicate rows: 96
Unique duplicated texts: 92
Conflicting-label duplicate texts: 0

TEST
----------------------------------------
Duplicate rows: 199
Unique duplicated texts: 191
Conflicting-label duplicate texts: 0

TRAIN vs TEST
----------------------------------------
Overlapping unique review texts: 123
Conflicting-label overlaps: 0

EXAMPLE OVERLAPPING REVIEWS
----------------------------------------

Example 1
Train label: 0
Test label : 0
Text: This movie was a rather odd viewing experience. The movie is obviously based on a play. Now I'm sure that everything in this movie works out just fine in a play but for in a movie it just doesn't feel terribly interesting enough to watch. The movie i ...

Example 2
Train label: 0
Test label : 0
Text: Quite what the producers of this appalling adaptation were trying to do is impossible to fathom.<br /><br />A group of top quality actors, in the main

In [11]:
import pandas as pd

# Load existing processed files

train_df = pd.read_csv("/content/data/processed/train_raw.csv")
test_df = pd.read_csv("/content/data/processed/test_raw.csv")

print("Train loaded:", train_df.shape)
print("Test loaded :", test_df.shape)

print("\n" + "=" * 60)
print("DUPLICATE & LEAKAGE INVESTIGATION")
print("=" * 60)

# ---------------- TRAIN DUPLICATES ----------------

train_dup_mask = train_df["text"].duplicated(keep=False)
train_dups = train_df[train_dup_mask]

train_conflicts = train_dups.groupby("text")["label"].nunique()

print("\nTRAIN")
print("-" * 40)
print("Duplicate rows:", train_df["text"].duplicated().sum())
print("Unique duplicated texts:", train_dups["text"].nunique())
print("Conflicting-label duplicate texts:",
(train_conflicts > 1).sum())

# ---------------- TEST DUPLICATES ----------------

test_dup_mask = test_df["text"].duplicated(keep=False)
test_dups = test_df[test_dup_mask]

test_conflicts = test_dups.groupby("text")["label"].nunique()

print("\nTEST")
print("-" * 40)
print("Duplicate rows:", test_df["text"].duplicated().sum())
print("Unique duplicated texts:", test_dups["text"].nunique())
print("Conflicting-label duplicate texts:",
(test_conflicts > 1).sum())

# ---------------- TRAIN / TEST LEAKAGE ----------------

train_unique = train_df.drop_duplicates(subset="text")
test_unique = test_df.drop_duplicates(subset="text")

overlap_texts = set(train_unique["text"]) & set(test_unique["text"])

print("\nTRAIN vs TEST")
print("-" * 40)
print("Overlapping unique review texts:", len(overlap_texts))

# Check whether overlapping reviews have different labels

train_overlap = train_unique[
train_unique["text"].isin(overlap_texts)
].set_index("text")

test_overlap = test_unique[
test_unique["text"].isin(overlap_texts)
].set_index("text")

common_texts = train_overlap.index.intersection(test_overlap.index)

conflicting_overlap = (
train_overlap.loc[common_texts, "label"].values
!= test_overlap.loc[common_texts, "label"].values
).sum()

print("Conflicting-label overlaps:", conflicting_overlap)

print("\n" + "=" * 60)
print("INVESTIGATION COMPLETE")
print("=" * 60)


Train loaded: (25000, 2)
Test loaded : (25000, 2)

DUPLICATE & LEAKAGE INVESTIGATION

TRAIN
----------------------------------------
Duplicate rows: 96
Unique duplicated texts: 92
Conflicting-label duplicate texts: 0

TEST
----------------------------------------
Duplicate rows: 199
Unique duplicated texts: 191
Conflicting-label duplicate texts: 0

TRAIN vs TEST
----------------------------------------
Overlapping unique review texts: 123
Conflicting-label overlaps: 0

INVESTIGATION COMPLETE


In [12]:
import pandas as pd
import re

# ============================================================
# STEP 2: DATA CLEANING
# ============================================================

# Load the raw datasets
train_df = pd.read_csv("/content/data/processed/train_raw.csv")
test_df = pd.read_csv("/content/data/processed/test_raw.csv")

print("Train loaded:", train_df.shape)
print("Test loaded :", test_df.shape)


# ------------------------------------------------------------
# 1. Remove duplicate reviews within each split
# ------------------------------------------------------------

train_before = len(train_df)
test_before = len(test_df)

train_duplicates = train_df["text"].duplicated().sum()
test_duplicates = test_df["text"].duplicated().sum()

train_df = train_df.drop_duplicates(
    subset="text",
    keep="first"
).reset_index(drop=True)

test_df = test_df.drop_duplicates(
    subset="text",
    keep="first"
).reset_index(drop=True)


# ------------------------------------------------------------
# 2. Remove train-test overlap from TEST
# ------------------------------------------------------------

train_texts = set(train_df["text"])

test_before_overlap = len(test_df)

test_df = test_df[
    ~test_df["text"].isin(train_texts)
].reset_index(drop=True)

overlap_removed = test_before_overlap - len(test_df)


# ------------------------------------------------------------
# 3. Clean the review text
# ------------------------------------------------------------

def clean_text(text):
    text = str(text)

    # Remove HTML tags such as <br />
    text = re.sub(r"<[^>]+>", " ", text)

    # Replace multiple spaces/newlines with one space
    text = re.sub(r"\s+", " ", text)

    # Remove spaces from beginning and end
    text = text.strip()

    return text


train_df["text"] = train_df["text"].apply(clean_text)
test_df["text"] = test_df["text"].apply(clean_text)


# ------------------------------------------------------------
# 4. Remove empty reviews after cleaning
# ------------------------------------------------------------

train_empty_before = len(train_df)

train_df = train_df[
    train_df["text"].str.strip() != ""
].reset_index(drop=True)

train_empty_removed = train_empty_before - len(train_df)


test_empty_before = len(test_df)

test_df = test_df[
    test_df["text"].str.strip() != ""
].reset_index(drop=True)

test_empty_removed = test_empty_before - len(test_df)


# ------------------------------------------------------------
# 5. Final quality checks
# ------------------------------------------------------------

remaining_train_duplicates = train_df["text"].duplicated().sum()
remaining_test_duplicates = test_df["text"].duplicated().sum()

remaining_overlap = len(
    set(train_df["text"]) & set(test_df["text"])
)

train_missing = train_df["text"].isna().sum()
test_missing = test_df["text"].isna().sum()

train_empty = (train_df["text"].str.strip() == "").sum()
test_empty = (test_df["text"].str.strip() == "").sum()


# ------------------------------------------------------------
# 6. Display final results
# ------------------------------------------------------------

print("\n" + "=" * 55)
print("FINAL DATA CLEANING REPORT")
print("=" * 55)

print("\nREMOVED")
print("-" * 55)
print("Train duplicates removed       :", train_duplicates)
print("Test duplicates removed        :", test_duplicates)
print("Train-test overlaps removed    :", overlap_removed)
print("Empty train reviews removed    :", train_empty_removed)
print("Empty test reviews removed     :", test_empty_removed)

print("\nFINAL DATASET SIZE")
print("-" * 55)
print("Train rows:", len(train_df))
print("Test rows :", len(test_df))

print("\nFINAL QUALITY CHECK")
print("-" * 55)
print("Train missing text             :", train_missing)
print("Test missing text              :", test_missing)
print("Train empty text               :", train_empty)
print("Test empty text                :", test_empty)
print("Remaining train duplicates     :", remaining_train_duplicates)
print("Remaining test duplicates      :", remaining_test_duplicates)
print("Remaining train-test overlap   :", remaining_overlap)


# ------------------------------------------------------------
# 7. Save cleaned datasets
# ------------------------------------------------------------

train_clean_path = "/content/data/processed/train_clean.csv"
test_clean_path = "/content/data/processed/test_clean.csv"

train_df.to_csv(train_clean_path, index=False)
test_df.to_csv(test_clean_path, index=False)


# ------------------------------------------------------------
# 8. Create data quality audit
# ------------------------------------------------------------

audit_df = pd.DataFrame({
    "metric": [
        "Initial train rows",
        "Initial test rows",
        "Train duplicates removed",
        "Test duplicates removed",
        "Train-test overlaps removed from test",
        "Empty train reviews removed",
        "Empty test reviews removed",
        "Final train rows",
        "Final test rows",
        "Remaining train duplicates",
        "Remaining test duplicates",
        "Remaining train-test overlap",
        "Train missing text",
        "Test missing text",
        "Train empty text",
        "Test empty text"
    ],
    "value": [
        train_before,
        test_before,
        train_duplicates,
        test_duplicates,
        overlap_removed,
        train_empty_removed,
        test_empty_removed,
        len(train_df),
        len(test_df),
        remaining_train_duplicates,
        remaining_test_duplicates,
        remaining_overlap,
        train_missing,
        test_missing,
        train_empty,
        test_empty
    ]
})

audit_path = "/content/data/processed/data_quality_audit.csv"

audit_df.to_csv(audit_path, index=False)


# ------------------------------------------------------------
# 9. Final confirmation
# ------------------------------------------------------------

print("\n" + "=" * 55)
print("FILES SAVED")
print("=" * 55)

print("✓", train_clean_path)
print("✓", test_clean_path)
print("✓", audit_path)

print("\n✅ DATA CLEANING COMPLETED SUCCESSFULLY")

Train loaded: (25000, 2)
Test loaded : (25000, 2)

FINAL DATA CLEANING REPORT

REMOVED
-------------------------------------------------------
Train duplicates removed       : 96
Test duplicates removed        : 199
Train-test overlaps removed    : 123
Empty train reviews removed    : 0
Empty test reviews removed     : 0

FINAL DATASET SIZE
-------------------------------------------------------
Train rows: 24904
Test rows : 24678

FINAL QUALITY CHECK
-------------------------------------------------------
Train missing text             : 0
Test missing text              : 0
Train empty text               : 0
Test empty text                : 0
Remaining train duplicates     : 2
Remaining test duplicates      : 2
Remaining train-test overlap   : 0

FILES SAVED
✓ /content/data/processed/train_clean.csv
✓ /content/data/processed/test_clean.csv
✓ /content/data/processed/data_quality_audit.csv

✅ DATA CLEANING COMPLETED SUCCESSFULLY


In [13]:
# Remove duplicates created by the cleaning process

train_before = len(train_df)
test_before = len(test_df)

train_df = train_df.drop_duplicates(
    subset="text",
    keep="first"
).reset_index(drop=True)

test_df = test_df.drop_duplicates(
    subset="text",
    keep="first"
).reset_index(drop=True)

print("Additional train duplicates removed:", train_before - len(train_df))
print("Additional test duplicates removed :", test_before - len(test_df))

print("\nFinal train rows:", len(train_df))
print("Final test rows :", len(test_df))

print("\nRemaining train duplicates:",
      train_df["text"].duplicated().sum())

print("Remaining test duplicates:",
      test_df["text"].duplicated().sum())

print("Remaining train-test overlap:",
      len(set(train_df["text"]) & set(test_df["text"])))

# Save corrected cleaned datasets
train_df.to_csv(
    "/content/data/processed/train_clean.csv",
    index=False
)

test_df.to_csv(
    "/content/data/processed/test_clean.csv",
    index=False
)

print("\n✅ Final cleaned datasets saved.")

Additional train duplicates removed: 2
Additional test duplicates removed : 2

Final train rows: 24902
Final test rows : 24676

Remaining train duplicates: 0
Remaining test duplicates: 0
Remaining train-test overlap: 0

✅ Final cleaned datasets saved.


In [14]:
# ============================================================
# STEP 3: TF-IDF FEATURE ENGINEERING
# ============================================================

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import save_npz
import joblib
import os

# ------------------------------------------------------------
# 1. Load cleaned datasets
# ------------------------------------------------------------

train_df = pd.read_csv("/content/data/processed/train_clean.csv")
test_df = pd.read_csv("/content/data/processed/test_clean.csv")

print("Train data:", train_df.shape)
print("Test data :", test_df.shape)


# ------------------------------------------------------------
# 2. Separate text and labels
# ------------------------------------------------------------

X_train_text = train_df["text"]
y_train = train_df["label"]

X_test_text = test_df["text"]
y_test = test_df["label"]


# ------------------------------------------------------------
# 3. Create TF-IDF vectorizer
# ------------------------------------------------------------

vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

print("\nFitting TF-IDF on TRAINING data only...")


# ------------------------------------------------------------
# 4. FIT ONLY on training data
# ------------------------------------------------------------

X_train_tfidf = vectorizer.fit_transform(X_train_text)


# ------------------------------------------------------------
# 5. Transform TEST using the same vectorizer
# ------------------------------------------------------------

X_test_tfidf = vectorizer.transform(X_test_text)


# ------------------------------------------------------------
# 6. Display feature information
# ------------------------------------------------------------

print("\n" + "=" * 55)
print("TF-IDF FEATURE REPORT")
print("=" * 55)

print("\nTraining feature matrix:")
print("Rows    :", X_train_tfidf.shape[0])
print("Features:", X_train_tfidf.shape[1])

print("\nTesting feature matrix:")
print("Rows    :", X_test_tfidf.shape[0])
print("Features:", X_test_tfidf.shape[1])

print("\nVocabulary size:", len(vectorizer.vocabulary_))

print("\nMatrix type:")
print(type(X_train_tfidf))

print("\nNon-zero training values:", X_train_tfidf.nnz)
print("Non-zero testing values :", X_test_tfidf.nnz)


# ------------------------------------------------------------
# 7. OOV explanation/check
# ------------------------------------------------------------

# Words appearing in TEST but not in the TRAIN vocabulary
train_vocabulary = set(vectorizer.vocabulary_.keys())

test_words = set()

for text in X_test_text:
    test_words.update(text.lower().split())

oov_words = test_words - train_vocabulary

print("\n" + "=" * 55)
print("OOV CHECK")
print("=" * 55)

print("Unique test tokens:", len(test_words))
print("OOV tokens       :", len(oov_words))

print("\nOOV handling:")
print("✓ Test vocabulary was NOT used to fit TF-IDF.")
print("✓ Unseen test terms are ignored by the fitted vectorizer.")
print("✓ No test information was used to create the vocabulary.")


# ------------------------------------------------------------
# 8. Save sparse feature matrices
# ------------------------------------------------------------

processed_path = "/content/data/processed"

train_features_path = os.path.join(
    processed_path,
    "train_tfidf.npz"
)

test_features_path = os.path.join(
    processed_path,
    "test_tfidf.npz"
)

save_npz(train_features_path, X_train_tfidf)
save_npz(test_features_path, X_test_tfidf)


# ------------------------------------------------------------
# 9. Save labels separately
# ------------------------------------------------------------

y_train.to_csv(
    os.path.join(processed_path, "train_labels.csv"),
    index=False
)

y_test.to_csv(
    os.path.join(processed_path, "test_labels.csv"),
    index=False
)


# ------------------------------------------------------------
# 10. Save the fitted vectorizer
# ------------------------------------------------------------

vectorizer_path = os.path.join(
    processed_path,
    "tfidf_vectorizer.joblib"
)

joblib.dump(vectorizer, vectorizer_path)


# ------------------------------------------------------------
# 11. Save feature names
# ------------------------------------------------------------

feature_names = vectorizer.get_feature_names_out()

pd.DataFrame({
    "feature": feature_names
}).to_csv(
    os.path.join(processed_path, "tfidf_features.csv"),
    index=False
)


# ------------------------------------------------------------
# 12. Final confirmation
# ------------------------------------------------------------

print("\n" + "=" * 55)
print("FILES SAVED")
print("=" * 55)

print("✓ train_tfidf.npz")
print("✓ test_tfidf.npz")
print("✓ train_labels.csv")
print("✓ test_labels.csv")
print("✓ tfidf_vectorizer.joblib")
print("✓ tfidf_features.csv")

print("\n✅ TF-IDF FEATURE ENGINEERING COMPLETED")

Train data: (24902, 2)
Test data : (24676, 2)

Fitting TF-IDF on TRAINING data only...

TF-IDF FEATURE REPORT

Training feature matrix:
Rows    : 24902
Features: 433653

Testing feature matrix:
Rows    : 24676
Features: 433653

Vocabulary size: 433653

Matrix type:
<class 'scipy.sparse._csr.csr_matrix'>

Non-zero training values: 7528088
Non-zero testing values : 6921225

OOV CHECK
Unique test tokens: 231465
OOV tokens       : 195560

OOV handling:
✓ Test vocabulary was NOT used to fit TF-IDF.
✓ Unseen test terms are ignored by the fitted vectorizer.
✓ No test information was used to create the vocabulary.

FILES SAVED
✓ train_tfidf.npz
✓ test_tfidf.npz
✓ train_labels.csv
✓ test_labels.csv
✓ tfidf_vectorizer.joblib
✓ tfidf_features.csv

✅ TF-IDF FEATURE ENGINEERING COMPLETED


In [15]:
# ============================================================
# STEP 4: CREATE src/preprocessing.py
# ============================================================

from pathlib import Path

script = r'''
import re
from pathlib import Path

import joblib
import pandas as pd
from scipy.sparse import save_npz
from sklearn.feature_extraction.text import TfidfVectorizer


# ============================================================
# CONFIGURATION
# ============================================================

BASE_DIR = Path(__file__).resolve().parent.parent

PROCESSED_DIR = BASE_DIR / "data" / "processed"

TRAIN_RAW = PROCESSED_DIR / "train_raw.csv"
TEST_RAW = PROCESSED_DIR / "test_raw.csv"

TRAIN_CLEAN = PROCESSED_DIR / "train_clean.csv"
TEST_CLEAN = PROCESSED_DIR / "test_clean.csv"

TRAIN_TFIDF = PROCESSED_DIR / "train_tfidf.npz"
TEST_TFIDF = PROCESSED_DIR / "test_tfidf.npz"

TRAIN_LABELS = PROCESSED_DIR / "train_labels.csv"
TEST_LABELS = PROCESSED_DIR / "test_labels.csv"

VECTORIZER_FILE = PROCESSED_DIR / "tfidf_vectorizer.joblib"
FEATURE_NAMES_FILE = PROCESSED_DIR / "tfidf_features.csv"

AUDIT_FILE = PROCESSED_DIR / "data_quality_audit.csv"


# ============================================================
# TEXT CLEANING
# ============================================================

def clean_text(text):
    """Remove HTML tags and normalize whitespace."""

    text = str(text)

    # Remove HTML tags such as <br />
    text = re.sub(r"<[^>]+>", " ", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()


# ============================================================
# LOAD DATA
# ============================================================

def load_data():
    """Load the raw train and test CSV files."""

    train_df = pd.read_csv(TRAIN_RAW)
    test_df = pd.read_csv(TEST_RAW)

    required_columns = {"text", "label"}

    if not required_columns.issubset(train_df.columns):
        raise ValueError(
            "Training data must contain 'text' and 'label' columns."
        )

    if not required_columns.issubset(test_df.columns):
        raise ValueError(
            "Test data must contain 'text' and 'label' columns."
        )

    return train_df, test_df


# ============================================================
# DATA CLEANING
# ============================================================

def clean_data(train_df, test_df):
    """Remove duplicates, leakage and clean review text."""

    initial_train_rows = len(train_df)
    initial_test_rows = len(test_df)

    # Remove missing text
    train_df = train_df.dropna(subset=["text"]).copy()
    test_df = test_df.dropna(subset=["text"]).copy()

    # Remove duplicate reviews within each split
    train_duplicates = train_df["text"].duplicated().sum()
    test_duplicates = test_df["text"].duplicated().sum()

    train_df = train_df.drop_duplicates(
        subset="text",
        keep="first"
    ).reset_index(drop=True)

    test_df = test_df.drop_duplicates(
        subset="text",
        keep="first"
    ).reset_index(drop=True)

    # Remove exact train-test overlap from TEST
    train_texts = set(train_df["text"])

    before_overlap = len(test_df)

    test_df = test_df[
        ~test_df["text"].isin(train_texts)
    ].reset_index(drop=True)

    overlap_removed = before_overlap - len(test_df)

    # Clean text
    train_df["text"] = train_df["text"].apply(clean_text)
    test_df["text"] = test_df["text"].apply(clean_text)

    # Remove empty reviews created after cleaning
    train_df = train_df[
        train_df["text"].str.strip() != ""
    ].reset_index(drop=True)

    test_df = test_df[
        test_df["text"].str.strip() != ""
    ].reset_index(drop=True)

    # Final duplicate check after cleaning
    train_df = train_df.drop_duplicates(
        subset="text",
        keep="first"
    ).reset_index(drop=True)

    test_df = test_df.drop_duplicates(
        subset="text",
        keep="first"
    ).reset_index(drop=True)

    # Final leakage check
    remaining_overlap = len(
        set(train_df["text"]) &
        set(test_df["text"])
    )

    if remaining_overlap != 0:
        raise ValueError(
            f"Data leakage detected: {remaining_overlap} "
            "train-test overlapping reviews remain."
        )

    audit = {
        "Initial train rows": initial_train_rows,
        "Initial test rows": initial_test_rows,
        "Train duplicates removed": train_duplicates,
        "Test duplicates removed": test_duplicates,
        "Train-test overlaps removed from test": overlap_removed,
        "Final train rows": len(train_df),
        "Final test rows": len(test_df),
        "Remaining train duplicates":
            train_df["text"].duplicated().sum(),
        "Remaining test duplicates":
            test_df["text"].duplicated().sum(),
        "Remaining train-test overlap": remaining_overlap,
        "Train missing text":
            train_df["text"].isna().sum(),
        "Test missing text":
            test_df["text"].isna().sum(),
        "Train empty text":
            (train_df["text"].str.strip() == "").sum(),
        "Test empty text":
            (test_df["text"].str.strip() == "").sum()
    }

    return train_df, test_df, audit


# ============================================================
# TF-IDF FEATURE ENGINEERING
# ============================================================

def create_tfidf(train_df, test_df):
    """
    Fit TF-IDF ONLY on training data.
    Transform test data using the same fitted vectorizer.
    """

    vectorizer = TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True
    )

    # IMPORTANT:
    # Vocabulary and IDF are learned ONLY from training data.
    X_train = vectorizer.fit_transform(train_df["text"])

    # Test data is transformed using the training vocabulary.
    X_test = vectorizer.transform(test_df["text"])

    return X_train, X_test, vectorizer


# ============================================================
# SAVE OUTPUTS
# ============================================================

def save_outputs(
    train_df,
    test_df,
    X_train,
    X_test,
    vectorizer,
    audit
):
    """Save cleaned data, features, labels and audit information."""

    PROCESSED_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    # Clean datasets
    train_df.to_csv(
        TRAIN_CLEAN,
        index=False
    )

    test_df.to_csv(
        TEST_CLEAN,
        index=False
    )

    # Sparse TF-IDF matrices
    save_npz(
        TRAIN_TFIDF,
        X_train
    )

    save_npz(
        TEST_TFIDF,
        X_test
    )

    # Labels
    train_df[["label"]].to_csv(
        TRAIN_LABELS,
        index=False
    )

    test_df[["label"]].to_csv(
        TEST_LABELS,
        index=False
    )

    # Vectorizer
    joblib.dump(
        vectorizer,
        VECTORIZER_FILE
    )

    # Feature names
    pd.DataFrame({
        "feature": vectorizer.get_feature_names_out()
    }).to_csv(
        FEATURE_NAMES_FILE,
        index=False
    )

    # Add TF-IDF information to audit
    audit["TF-IDF features"] = X_train.shape[1]
    audit["Training matrix rows"] = X_train.shape[0]
    audit["Testing matrix rows"] = X_test.shape[0]
    audit["TF-IDF fitted on"] = "Training data only"
    audit["Test transformation"] = "Training-fitted vectorizer"
    audit["Feature matrix format"] = "Sparse CSR / NPZ"

    pd.DataFrame(
        list(audit.items()),
        columns=["metric", "value"]
    ).to_csv(
        AUDIT_FILE,
        index=False
    )


# ============================================================
# MAIN PIPELINE
# ============================================================

def main():

    print("=" * 60)
    print("IMDb DATA ENGINEERING PIPELINE")
    print("=" * 60)

    print("\n1. Loading data...")
    train_df, test_df = load_data()

    print("Initial train rows:", len(train_df))
    print("Initial test rows :", len(test_df))

    print("\n2. Cleaning and leakage prevention...")
    train_df, test_df, audit = clean_data(
        train_df,
        test_df
    )

    print("Final train rows:", len(train_df))
    print("Final test rows :", len(test_df))

    print("\n3. Creating TF-IDF features...")
    X_train, X_test, vectorizer = create_tfidf(
        train_df,
        test_df
    )

    print("TF-IDF features:", X_train.shape[1])
    print("Train matrix:", X_train.shape)
    print("Test matrix :", X_test.shape)

    print("\n4. Saving outputs...")
    save_outputs(
        train_df,
        test_df,
        X_train,
        X_test,
        vectorizer,
        audit
    )

    print("\n" + "=" * 60)
    print("PIPELINE COMPLETED SUCCESSFULLY")
    print("=" * 60)


if __name__ == "__main__":
    main()
'''

# Create src directory
Path("/content/src").mkdir(parents=True, exist_ok=True)

# Write preprocessing.py
file_path = Path("/content/src/preprocessing.py")
file_path.write_text(script, encoding="utf-8")

print("✅ preprocessing.py created successfully")
print(file_path)

✅ preprocessing.py created successfully
/content/src/preprocessing.py


In [16]:
from pathlib import Path

path = Path("/content/src/preprocessing.py")

print("File exists:", path.exists())
print("File size:", path.stat().st_size, "bytes")

print("\nFirst 10 lines:")
print("\n".join(path.read_text().splitlines()[:10]))

File exists: True
File size: 8638 bytes

First 10 lines:

import re
from pathlib import Path

import joblib
import pandas as pd
from scipy.sparse import save_npz
from sklearn.feature_extraction.text import TfidfVectorizer




In [17]:
from pathlib import Path

processed = Path("/content/data/processed")

files = [
    "train_clean.csv",
    "test_clean.csv",
    "train_tfidf.npz",
    "test_tfidf.npz",
    "train_labels.csv",
    "test_labels.csv",
    "tfidf_vectorizer.joblib",
    "tfidf_features.csv",
    "data_quality_audit.csv",
]

print("CURRENT PROCESSED FILE SIZES")
print("=" * 60)

for name in files:
    path = processed / name

    if path.exists():
        size_mb = path.stat().st_size / (1024 * 1024)
        print(f"{name:<30} {size_mb:>10.2f} MB")
    else:
        print(f"{name:<30} NOT FOUND")

CURRENT PROCESSED FILE SIZES
train_clean.csv                     31.15 MB
test_clean.csv                      30.12 MB
train_tfidf.npz                     61.96 MB
test_tfidf.npz                      56.77 MB
train_labels.csv                     0.05 MB
test_labels.csv                      0.05 MB
tfidf_vectorizer.joblib             10.97 MB
tfidf_features.csv                   4.88 MB
data_quality_audit.csv               0.00 MB


In [18]:
from pathlib import Path

path = Path("/content/src/preprocessing.py")

final_code = r'''
import re
from pathlib import Path

import joblib
import pandas as pd
from scipy.sparse import save_npz
from sklearn.feature_extraction.text import TfidfVectorizer


# ============================================================
# PATH CONFIGURATION
# ============================================================

BASE_DIR = Path(__file__).resolve().parent.parent
PROCESSED_DIR = BASE_DIR / "data" / "processed"

TRAIN_RAW = PROCESSED_DIR / "train_raw.csv"
TEST_RAW = PROCESSED_DIR / "test_raw.csv"

TRAIN_CLEAN = PROCESSED_DIR / "train_clean.csv"
TEST_CLEAN = PROCESSED_DIR / "test_clean.csv"

TRAIN_TFIDF = PROCESSED_DIR / "train_tfidf.npz"
TEST_TFIDF = PROCESSED_DIR / "test_tfidf.npz"

TRAIN_LABELS = PROCESSED_DIR / "train_labels.csv"
TEST_LABELS = PROCESSED_DIR / "test_labels.csv"

VECTORIZER_FILE = PROCESSED_DIR / "tfidf_vectorizer.joblib"
FEATURE_NAMES_FILE = PROCESSED_DIR / "tfidf_features.csv"

AUDIT_FILE = PROCESSED_DIR / "data_quality_audit.csv"


# ============================================================
# TEXT CLEANING
# ============================================================

def clean_text(text):
    """
    Remove HTML tags and normalize whitespace.
    """
    text = str(text)

    # Remove HTML tags such as <br />
    text = re.sub(r"<[^>]+>", " ", text)

    # Normalize multiple spaces/newlines
    text = re.sub(r"\s+", " ", text)

    return text.strip()


# ============================================================
# DATA LOADING AND VALIDATION
# ============================================================

def load_data():
    """
    Load the raw train and test CSV files and validate
    the required columns.
    """

    if not TRAIN_RAW.exists():
        raise FileNotFoundError(
            f"Training file not found: {TRAIN_RAW}"
        )

    if not TEST_RAW.exists():
        raise FileNotFoundError(
            f"Test file not found: {TEST_RAW}"
        )

    train_df = pd.read_csv(TRAIN_RAW)
    test_df = pd.read_csv(TEST_RAW)

    required_columns = {"text", "label"}

    if not required_columns.issubset(train_df.columns):
        raise ValueError(
            "Training data must contain 'text' and 'label' columns."
        )

    if not required_columns.issubset(test_df.columns):
        raise ValueError(
            "Test data must contain 'text' and 'label' columns."
        )

    return train_df, test_df


# ============================================================
# DATA CLEANING + LEAKAGE PREVENTION
# ============================================================

def clean_data(train_df, test_df):

    audit = {}

    # --------------------------------------------------------
    # Initial row counts
    # --------------------------------------------------------

    audit["Initial train rows"] = len(train_df)
    audit["Initial test rows"] = len(test_df)

    # --------------------------------------------------------
    # Missing text values
    # --------------------------------------------------------

    train_missing = train_df["text"].isna().sum()
    test_missing = test_df["text"].isna().sum()

    audit["Train missing text"] = train_missing
    audit["Test missing text"] = test_missing

    train_df = train_df.dropna(subset=["text"]).copy()
    test_df = test_df.dropna(subset=["text"]).copy()

    # --------------------------------------------------------
    # Clean text FIRST
    # --------------------------------------------------------

    train_df["text"] = train_df["text"].apply(clean_text)
    test_df["text"] = test_df["text"].apply(clean_text)

    # --------------------------------------------------------
    # Remove empty reviews
    # --------------------------------------------------------

    train_empty = (train_df["text"].str.strip() == "").sum()
    test_empty = (test_df["text"].str.strip() == "").sum()

    audit["Train empty text removed"] = train_empty
    audit["Test empty text removed"] = test_empty

    train_df = train_df[
        train_df["text"].str.strip() != ""
    ].copy()

    test_df = test_df[
        test_df["text"].str.strip() != ""
    ].copy()

    # --------------------------------------------------------
    # Remove duplicate reviews WITHIN each split
    # --------------------------------------------------------

    train_duplicates = train_df["text"].duplicated().sum()
    test_duplicates = test_df["text"].duplicated().sum()

    audit["Train duplicates removed"] = train_duplicates
    audit["Test duplicates removed"] = test_duplicates

    train_df = train_df.drop_duplicates(
        subset="text",
        keep="first"
    ).reset_index(drop=True)

    test_df = test_df.drop_duplicates(
        subset="text",
        keep="first"
    ).reset_index(drop=True)

    # --------------------------------------------------------
    # Check for train/test overlap AFTER cleaning
    # --------------------------------------------------------

    train_texts = set(train_df["text"])

    overlap_count = test_df["text"].isin(train_texts).sum()

    audit["Train-test overlaps removed from test"] = overlap_count

    if overlap_count > 0:
        test_df = test_df[
            ~test_df["text"].isin(train_texts)
        ].reset_index(drop=True)

    # --------------------------------------------------------
    # Final duplicate and leakage validation
    # --------------------------------------------------------

    remaining_train_duplicates = (
        train_df["text"].duplicated().sum()
    )

    remaining_test_duplicates = (
        test_df["text"].duplicated().sum()
    )

    remaining_overlap = len(
        set(train_df["text"]) &
        set(test_df["text"])
    )

    audit["Remaining train duplicates"] = (
        remaining_train_duplicates
    )

    audit["Remaining test duplicates"] = (
        remaining_test_duplicates
    )

    audit["Remaining train-test overlap"] = (
        remaining_overlap
    )

    # Hard safety check
    if remaining_train_duplicates != 0:
        raise ValueError(
            "Data quality error: duplicate reviews remain in training data."
        )

    if remaining_test_duplicates != 0:
        raise ValueError(
            "Data quality error: duplicate reviews remain in test data."
        )

    if remaining_overlap != 0:
        raise ValueError(
            "DATA LEAKAGE DETECTED: train/test reviews overlap."
        )

    # --------------------------------------------------------
    # Final row counts
    # --------------------------------------------------------

    audit["Final train rows"] = len(train_df)
    audit["Final test rows"] = len(test_df)

    # --------------------------------------------------------
    # Label distribution
    # --------------------------------------------------------

    audit["Train positive reviews"] = int(
        (train_df["label"] == 1).sum()
    )

    audit["Train negative reviews"] = int(
        (train_df["label"] == 0).sum()
    )

    audit["Test positive reviews"] = int(
        (test_df["label"] == 1).sum()
    )

    audit["Test negative reviews"] = int(
        (test_df["label"] == 0).sum()
    )

    return train_df, test_df, audit


# ============================================================
# TF-IDF FEATURE ENGINEERING
# ============================================================

def create_tfidf(train_df, test_df):

    vectorizer = TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True,
        dtype="float32"
    )

    # IMPORTANT:
    # Fit ONLY on training data.
    X_train = vectorizer.fit_transform(
        train_df["text"]
    )

    # Transform test data using the training-fitted vectorizer.
    X_test = vectorizer.transform(
        test_df["text"]
    )

    return X_train, X_test, vectorizer


# ============================================================
# SAVE OUTPUTS
# ============================================================

def save_outputs(
    train_df,
    test_df,
    X_train,
    X_test,
    vectorizer,
    audit
):

    PROCESSED_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    # Clean datasets
    train_df.to_csv(
        TRAIN_CLEAN,
        index=False
    )

    test_df.to_csv(
        TEST_CLEAN,
        index=False
    )

    # Sparse TF-IDF matrices
    save_npz(
        TRAIN_TFIDF,
        X_train
    )

    save_npz(
        TEST_TFIDF,
        X_test
    )

    # Labels
    train_df[["label"]].to_csv(
        TRAIN_LABELS,
        index=False
    )

    test_df[["label"]].to_csv(
        TEST_LABELS,
        index=False
    )

    # Vectorizer
    joblib.dump(
        vectorizer,
        VECTORIZER_FILE
    )

    # Feature names
    pd.DataFrame({
        "feature": vectorizer.get_feature_names_out()
    }).to_csv(
        FEATURE_NAMES_FILE,
        index=False
    )

    # --------------------------------------------------------
    # TF-IDF audit information
    # --------------------------------------------------------

    audit["TF-IDF features"] = X_train.shape[1]

    audit["Training matrix rows"] = X_train.shape[0]
    audit["Training matrix columns"] = X_train.shape[1]

    audit["Testing matrix rows"] = X_test.shape[0]
    audit["Testing matrix columns"] = X_test.shape[1]

    audit["Training non-zero values"] = X_train.nnz
    audit["Testing non-zero values"] = X_test.nnz

    audit["TF-IDF fitted on"] = (
        "Training data only"
    )

    audit["Test transformation"] = (
        "Training-fitted vectorizer"
    )

    audit["TF-IDF data type"] = "float32"

    audit["Feature matrix format"] = (
        "Sparse CSR / NPZ"
    )

    # --------------------------------------------------------
    # Save audit
    # --------------------------------------------------------

    audit_df = pd.DataFrame(
        list(audit.items()),
        columns=["metric", "value"]
    )

    audit_df.to_csv(
        AUDIT_FILE,
        index=False
    )


# ============================================================
# MAIN PIPELINE
# ============================================================

def main():

    print("=" * 65)
    print("IMDb DATA ENGINEERING PIPELINE")
    print("=" * 65)

    # --------------------------------------------------------
    # 1. Load
    # --------------------------------------------------------

    print("\n[1/4] Loading raw data...")

    train_df, test_df = load_data()

    print(
        f"Initial train rows: {len(train_df):,}"
    )

    print(
        f"Initial test rows : {len(test_df):,}"
    )

    # --------------------------------------------------------
    # 2. Clean
    # --------------------------------------------------------

    print("\n[2/4] Cleaning data and preventing leakage...")

    train_df, test_df, audit = clean_data(
        train_df,
        test_df
    )

    print(
        f"Final train rows: {len(train_df):,}"
    )

    print(
        f"Final test rows : {len(test_df):,}"
    )

    print(
        "Train/test overlap: 0"
    )

    # --------------------------------------------------------
    # 3. TF-IDF
    # --------------------------------------------------------

    print("\n[3/4] Creating TF-IDF features...")

    X_train, X_test, vectorizer = create_tfidf(
        train_df,
        test_df
    )

    print(
        f"Vocabulary size: {X_train.shape[1]:,}"
    )

    print(
        f"Train matrix: {X_train.shape}"
    )

    print(
        f"Test matrix : {X_test.shape}"
    )

    print(
        f"Train non-zero values: {X_train.nnz:,}"
    )

    print(
        f"Test non-zero values : {X_test.nnz:,}"
    )

    # --------------------------------------------------------
    # 4. Save
    # --------------------------------------------------------

    print("\n[4/4] Saving processed outputs...")

    save_outputs(
        train_df,
        test_df,
        X_train,
        X_test,
        vectorizer,
        audit
    )

    print("\n" + "=" * 65)
    print("PIPELINE COMPLETED SUCCESSFULLY")
    print("=" * 65)

    print("\nGenerated files:")

    for file in [
        TRAIN_CLEAN,
        TEST_CLEAN,
        TRAIN_TFIDF,
        TEST_TFIDF,
        TRAIN_LABELS,
        TEST_LABELS,
        VECTORIZER_FILE,
        FEATURE_NAMES_FILE,
        AUDIT_FILE
    ]:
        print(f"  ✓ {file.name}")


if __name__ == "__main__":
    main()
'''

path.write_text(final_code, encoding="utf-8")

print("Final preprocessing.py written successfully.")
print("File size:", path.stat().st_size, "bytes")

Final preprocessing.py written successfully.
File size: 12376 bytes


In [19]:
from pathlib import Path

path = Path("/content/src/preprocessing.py")

print("Exists:", path.exists())
print("Size:", path.stat().st_size, "bytes")

print("\nLast 10 lines:")
print("\n".join(path.read_text().splitlines()[-10:]))

Exists: True
Size: 12376 bytes

Last 10 lines:
        TEST_LABELS,
        VECTORIZER_FILE,
        FEATURE_NAMES_FILE,
        AUDIT_FILE
    ]:
        print(f"  ✓ {file.name}")


if __name__ == "__main__":
    main()


In [20]:
import sys
from pathlib import Path

sys.path.insert(0, "/content/src")

from preprocessing import main

main()

IMDb DATA ENGINEERING PIPELINE

[1/4] Loading raw data...
Initial train rows: 25,000
Initial test rows : 25,000

[2/4] Cleaning data and preventing leakage...
Final train rows: 24,902
Final test rows : 24,676
Train/test overlap: 0

[3/4] Creating TF-IDF features...


/usr/local/lib/python3.13/dist-packages/sklearn/feature_extraction/text.py:2043: UserWarning: Only (<class 'numpy.float64'>, <class 'numpy.float32'>, <class 'numpy.float16'>) 'dtype' should be used. float32 'dtype' will be converted to np.float64.
  warnings.warn(


Vocabulary size: 433,653
Train matrix: (24902, 433653)
Test matrix : (24676, 433653)
Train non-zero values: 7,528,088
Test non-zero values : 6,921,225

[4/4] Saving processed outputs...

PIPELINE COMPLETED SUCCESSFULLY

Generated files:
  ✓ train_clean.csv
  ✓ test_clean.csv
  ✓ train_tfidf.npz
  ✓ test_tfidf.npz
  ✓ train_labels.csv
  ✓ test_labels.csv
  ✓ tfidf_vectorizer.joblib
  ✓ tfidf_features.csv
  ✓ data_quality_audit.csv


In [21]:
from pathlib import Path
import pandas as pd

processed = Path("/content/data/processed")

print("=" * 65)
print("FINAL DATA ENGINEERING VERIFICATION")
print("=" * 65)

# ------------------------------------------------------------
# File sizes
# ------------------------------------------------------------

print("\nFILE SIZES")
print("-" * 65)

for name in [
    "train_clean.csv",
    "test_clean.csv",
    "train_tfidf.npz",
    "test_tfidf.npz",
    "train_labels.csv",
    "test_labels.csv",
    "tfidf_vectorizer.joblib",
    "tfidf_features.csv",
    "data_quality_audit.csv",
]:
    path = processed / name

    if path.exists():
        size_mb = path.stat().st_size / (1024 * 1024)
        print(f"{name:<30} {size_mb:>10.2f} MB")
    else:
        print(f"{name:<30} NOT FOUND")

# ------------------------------------------------------------
# Audit
# ------------------------------------------------------------

print("\nDATA QUALITY AUDIT")
print("-" * 65)

audit = pd.read_csv(
    processed / "data_quality_audit.csv"
)

print(audit.to_string(index=False))

FINAL DATA ENGINEERING VERIFICATION

FILE SIZES
-----------------------------------------------------------------
train_clean.csv                     31.15 MB
test_clean.csv                      30.12 MB
train_tfidf.npz                     40.14 MB
test_tfidf.npz                      35.85 MB
train_labels.csv                     0.05 MB
test_labels.csv                      0.05 MB
tfidf_vectorizer.joblib              9.31 MB
tfidf_features.csv                   4.88 MB
data_quality_audit.csv               0.00 MB

DATA QUALITY AUDIT
-----------------------------------------------------------------
                               metric                      value
                   Initial train rows                      25000
                    Initial test rows                      25000
                   Train missing text                          0
                    Test missing text                          0
             Train empty text removed                          0
     

In [22]:
from pathlib import Path

print("PROJECT FILES")
print("=" * 60)

for folder in [
    Path("/content/src"),
    Path("/content/data/processed")
]:
    print(f"\n📁 {folder}")
    for file in sorted(folder.iterdir()):
        if file.is_file():
            size_mb = file.stat().st_size / (1024 * 1024)
            print(f"  ✓ {file.name:<30} {size_mb:.2f} MB")

PROJECT FILES

📁 /content/src
  ✓ preprocessing.py               0.01 MB

📁 /content/data/processed
  ✓ data_quality_audit.csv         0.00 MB
  ✓ test_clean.csv                 30.12 MB
  ✓ test_labels.csv                0.05 MB
  ✓ test_raw.csv                   31.03 MB
  ✓ test_tfidf.npz                 35.85 MB
  ✓ tfidf_features.csv             4.88 MB
  ✓ tfidf_vectorizer.joblib        9.31 MB
  ✓ train_clean.csv                31.15 MB
  ✓ train_labels.csv               0.05 MB
  ✓ train_raw.csv                  31.78 MB
  ✓ train_tfidf.npz                40.14 MB


In [23]:
import shutil
from pathlib import Path

processed = Path("/content/data/processed")

files_to_upload = [
    "train_clean.csv",
    "test_clean.csv",
    "train_labels.csv",
    "test_labels.csv",
    "tfidf_features.csv",
    "data_quality_audit.csv",
    "train_tfidf.npz",
    "test_tfidf.npz",
    "tfidf_vectorizer.joblib",
]

upload_dir = Path("/content/data_engineer_upload")
upload_dir.mkdir(exist_ok=True)

for filename in files_to_upload:
    src = processed / filename
    if src.exists():
        shutil.copy2(src, upload_dir / filename)
        print("✓", filename)
    else:
        print("⚠️ Missing:", filename)

zip_path = shutil.make_archive(
    "/content/data_engineer_processed",
    "zip",
    upload_dir
)

print("\nZIP created:", zip_path)

✓ train_clean.csv
✓ test_clean.csv
✓ train_labels.csv
✓ test_labels.csv
✓ tfidf_features.csv
✓ data_quality_audit.csv
✓ train_tfidf.npz
✓ test_tfidf.npz
✓ tfidf_vectorizer.joblib

ZIP created: /content/data_engineer_processed.zip


In [1]:
from pathlib import Path

print("Searching /content for our Data Engineer files...\n")

targets = {
    "train_clean.csv",
    "test_clean.csv",
    "train_labels.csv",
    "test_labels.csv",
    "train_tfidf.npz",
    "test_tfidf.npz",
    "tfidf_vectorizer.joblib",
    "tfidf_features.csv",
    "data_quality_audit.csv",
}

found = []

for path in Path("/content").rglob("*"):
    if path.is_file() and path.name in targets:
        found.append(path)

if found:
    for path in found:
        size_mb = path.stat().st_size / (1024 * 1024)
        print(f"✓ {path}  ({size_mb:.2f} MB)")
else:
    print("❌ None of the processed files were found.")

Searching /content for our Data Engineer files...

❌ None of the processed files were found.


In [25]:
from google.colab import files

files.download("/content/data_engineer_processed.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>